**Batch Collection API Test**

In [ ]:
import os
from dotenv import load_dotenv


# Read local root .env configurations
load_dotenv()

print("OpenAI Key:", "Loaded" if os.getenv("OPENAI_API_KEY") else "Missing")
print("Anthropic Key:", "Loaded" if os.getenv("ANTHROPIC_API_KEY") else "Missing")
print("Gemini Key:", "Loaded" if os.getenv("GEMINI_API_KEY") else "Missing")

In [ ]:
from openai import OpenAI

client = OpenAI()
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Ping"}]
)
print("OpenAI Success:", response.choices[0].message.content)

In [ ]:
import anthropic

client = anthropic.Anthropic()
response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=2048,
    messages=[{"role": "user", "content": "Ping"}]
)

print("Claude Success:", response.content[0].text)

In [ ]:
from google import genai

client = genai.Client()
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Ping"
)
print("Gemini Success:", response.text)

__LLM Judge API Test__

In [ ]:
import os
from pathlib import Path
import yaml
from dotenv import load_dotenv
from src.cross_audit.llm_judge import call_judge_api

load_dotenv(override=True)

# Validate YAML parsing and default settings
with open("configs/judge_config.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

settings = cfg["judge_settings"]
default_key = settings["default_judge"]
judge_cfg = cfg["judges"][default_key]

# 2. Validate Codebook file and API Key existence
cb_path = Path(settings["codebook_path"])
api_key = os.getenv(judge_cfg["api_key_env"], "").strip()

assert cb_path.exists(), f"Codebook missing: {cb_path}"
assert len(api_key) > 20, f"Invalid/missing API key: {judge_cfg['api_key_env']}"
print(f"[OK] YAML Loaded | Default Judge: {default_key} ({judge_cfg['model_id']})")
print(f"[OK] Codebook Found: {cb_path} ({cb_path.stat().st_size} bytes)")
print(f"[OK] API Key Loaded: {judge_cfg['api_key_env']} (prefix={api_key[:12]}...)")


In [ ]:
# 1-token minimal connectivity test 
# validate model ID is valid and accessible)
ping_resp = call_judge_api(
    provider=judge_cfg["provider"],
    model_id=judge_cfg["model_id"],
    api_key=api_key,
    params={"temperature": 0.0, "max_tokens": 32},
    system_prompt="Return valid JSON only.",
    user_prompt='Reply with {"status": "ready"}',
)
print(f"[OK] API Ping Response: {ping_resp.strip()}")